In [1]:
from google.colab import files
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

df = pd.read_excel(nombre_archivo, sheet_name=0)
df = df.dropna(axis=1, how="all")   # quita columnas vacías
df.columns = [str(c).strip() for c in df.columns]

col_fecha = [c for c in df.columns if "Fecha" in c][0]
df[col_fecha] = pd.to_datetime(df[col_fecha])
df = df.sort_values(col_fecha).reset_index(drop=True)

cols_numeros = [c for c in df.columns if "Número" in c and "Sorteo" not in c]

print(f"{len(df)} sorteos cargados, del {df[col_fecha].min().date()} al {df[col_fecha].max().date()}")
df.head()

#Convertir a formato tabla binaria

rango_numeros = list(range(100))
binaria = pd.DataFrame(0, index=df.index, columns=rango_numeros)

for c in cols_numeros:
    for idx, val in df[c].items():
        if pd.notna(val):
            binaria.loc[idx, int(val)] = 1

binaria.head()


#Crear las variables
VENTANA = 20
N_SORTEOS = 500  # usamos solo los últimos 150 sorteos para que corra rápido
muestra = binaria.tail(N_SORTEOS).reset_index(drop=True)
X = pd.DataFrame({
    f"freq_{num}": muestra[num].rolling(VENTANA, min_periods=1).mean().shift(1).fillna(0)
    for num in rango_numeros
})
y = muestra
print("X (variables):", X.shape)
print("y (objetivo):", y.shape)

#4 entrenar al modelo

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

modelo = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)

#Evaluar el modelo
from sklearn.metrics import accuracy_score
exactitud_modelo = accuracy_score(y_test.values.ravel(), pred.ravel())
exactitud_azar = 3 / 100

print(f"Exactitud del Modelo : {exactitud_modelo:.2%}")
print(f"Exactitud esperada solo por azar: {exactitud_azar:.2%}")

# para poder dar el resultado de imprimir el número predictivo

import pandas as pd
import numpy as np

# Tomamos los datos más recientes

ultimo_sorteo_X = X.tail(1).reset_index(drop=True)  # reset de índice/ esa fila se presenta en la posición 0

# Usamos las probabilidades que salga uno de los 100 números (0 a 99)
# modelo.predict_proba devuelve una lista de 100 arreglos
probabilidades = []
for n in rango_numeros:
    # Probabilidad de que el número que salga favorecido sea 1
    prob_uno = modelo.estimators_[n].predict_proba(ultimo_sorteo_X)[0][1] #modelo.estimators_[n]: clasificador multisalida (MultiOutputClassifier)/ analiza cada número por separado
    probabilidades.append((n, prob_uno))

#Creamos el DataFrame / ordenados de mayor a menor
df_probabilidades = pd.DataFrame(probabilidades, columns=["numero", "probabilidad"])
df_probabilidades = df_probabilidades.sort_values(by="probabilidad", ascending=False).reset_index(drop=True)

# Usamos el número con la probabilidad más alta
numero_recomendado = df_probabilidades.loc[0, "numero"]
prob_recomendada = df_probabilidades.loc[0, "probabilidad"]

print("****************************************")
print(f"   Número más Probable de la Loteria: {numero_recomendado}")
print(f"   Probabilidad estimada: {prob_recomendada:.2%}")
print("****************************************")
print("\n Los 10 números más probables de la lotería a salir:")
print("numero  probabilidad")

# Calcular y mostrar los 10 primeros números con formato  decimal
for i in range(10):
    num = df_probabilidades.loc[i, "numero"] #loc[0, ...] busca la primera fila de una tabla (el índice 0).
    prob = df_probabilidades.loc[i, "probabilidad"]
    print(f"   {num:<5}  {prob:.3f}")  #Muestra las probabilidades con formato de 3 decimales


Saving Numeros_Favoreecidos- profe.xlsx to Numeros_Favoreecidos- profe.xlsx
569 sorteos cargados, del 2019-01-04 al 2025-12-02
X (variables): (500, 100)
y (objetivo): (500, 100)
Exactitud del Modelo : 97.02%
Exactitud esperada solo por azar: 3.00%
****************************************
   Número más Probable de la Loteria: 56
   Probabilidad estimada: 16.27%
****************************************

 Los 10 números más probables de la lotería a salir:
numero  probabilidad
   56     0.163
   33     0.139
   64     0.128
   59     0.112
   67     0.111
   94     0.104
   7      0.104
   73     0.103
   97     0.103
   87     0.095
